# Experimento MULTICLASE — identificar a qué generador pertenece un rostro

(Flux / SDXL / StyleGAN2 / StyleGAN3 / real), no solo fake vs real.

A diferencia de los experimentos A-D (binarios: fake vs real de UN generador a la vez), este experimento entrena UN modelo de 5 clases sobre las particiones balanceadas de `Particiones/Experimentos/Multiclase/` (creadas por `Crear_Particiones_Multiclase.py`), donde "real" combina FFHQ + CelebA-HQ a partes iguales. Es la base del clasificador que alimenta la página web de identificación de modelo (ver `webapp/`).

A diferencia del Experimento A, aquí se entrena UNA sola arquitectura (EfficientNet-B0, la misma que ya usás como base en el resto de experimentos) con UNA sola semilla (42): es una primera corrida rápida para tener un modelo funcionando (y poder probar la página web) sin pagar el costo de comparar arquitecturas todavía. Usa el mismo `nucleo.py` real (no una copia). Reanudable: si Colab se cae, re-ejecutar la última celda salta el entrenamiento si ya terminó.

### Antes de correr, sube a Drive:
```
MyDrive/Tesis/
    datos/config.py      <-- copia tal cual de tu carpeta local (Datasets/)
    datos/nucleo.py      <-- copia tal cual de tu carpeta local (Datasets/)
    Multiclase.zip        <-- zip de Particiones/Experimentos/Multiclase/
                               (debe contener train/, val/, test/, cada uno
                               con las carpetas Flux/ SDXL/ StyleGAN2/
                               StyleGAN3/ real/)
    Resultados2/           <-- se crea sola (o la que ya usas para el resto)
```


In [ ]:
!nvidia-smi


In [ ]:
!pip install timm --quiet
print("Listo")


In [ ]:
from google.colab import drive
import os, time, zipfile

drive.mount('/content/drive')

RUTA_ZIP_DRIVE = '/content/drive/MyDrive/Tesis/Multiclase.zip'  # <-- AJUSTA
RUTA_DATOS_LOCAL = '/content/Multiclase'  # disco local de la sesión (rápido)
CLASES_ESPERADAS = ["Flux", "SDXL", "StyleGAN2", "StyleGAN3", "real"]

if not os.path.exists(RUTA_ZIP_DRIVE):
    raise FileNotFoundError(
        f"No encuentro {RUTA_ZIP_DRIVE}\n"
        f"Sube el zip de Particiones/Experimentos/Multiclase a esa ruta en Drive."
    )

if not os.path.isdir(os.path.join(RUTA_DATOS_LOCAL, "train")):
    print("Descomprimiendo el dataset al disco local de Colab...")
    t0 = time.time()
    with zipfile.ZipFile(RUTA_ZIP_DRIVE, 'r') as z:
        z.extractall(RUTA_DATOS_LOCAL)
    print(f"Descomprimido en {time.time()-t0:.0f}s")
else:
    print("El dataset ya está descomprimido.")

# Según cómo se haya comprimido, la carpeta puede quedar anidada
# (p. ej. /content/Multiclase/Multiclase/train). Si es así, se corrige.
if not os.path.isdir(os.path.join(RUTA_DATOS_LOCAL, "train")):
    import shutil, glob
    candidatos = glob.glob(os.path.join(RUTA_DATOS_LOCAL, "**", "train"), recursive=True)
    if not candidatos:
        raise FileNotFoundError(
            f"Tras descomprimir no encuentro 'train/' dentro de {RUTA_DATOS_LOCAL}. "
            f"Contenido: {os.listdir(RUTA_DATOS_LOCAL)}"
        )
    origen = os.path.dirname(candidatos[0])
    if os.path.abspath(origen) != os.path.abspath(RUTA_DATOS_LOCAL):
        for item in os.listdir(origen):
            shutil.move(os.path.join(origen, item), os.path.join(RUTA_DATOS_LOCAL, item))

# Verificación: las 5 clases deben estar en los 3 splits
print("\nRuta de datos:", RUTA_DATOS_LOCAL)
total = 0
for split in ["train", "val", "test"]:
    for clase in CLASES_ESPERADAS:
        carpeta = os.path.join(RUTA_DATOS_LOCAL, split, clase)
        if not os.path.isdir(carpeta):
            raise FileNotFoundError(f"Falta la carpeta {carpeta}")
        n = len(os.listdir(carpeta))
        total += n
        print(f"  {split}/{clase}: {n:,} imágenes")
print(f"  TOTAL: {total:,} imágenes")


In [ ]:
import shutil, sys
from pathlib import Path

RUTA_CODIGO_DRIVE = Path('/content/drive/MyDrive/Tesis/codigo')  # <-- AJUSTA

for nombre in ['config.py', 'nucleo.py']:
    origen = RUTA_CODIGO_DRIVE / nombre
    if not origen.exists():
        raise FileNotFoundError(
            f"No encuentro {origen}\n"
            f"Sube tus {nombre} de Datasets/ a {RUTA_CODIGO_DRIVE}/"
        )
    shutil.copy(origen, Path('/content') / nombre)

sys.path.insert(0, '/content')
import config
import nucleo

if not hasattr(config, "CLASES_MULTICLASE"):
    raise AttributeError(
        "Tu config.py de Drive no tiene CLASES_MULTICLASE: es una versión "
        "vieja. Vuelve a subir el config.py actualizado de Datasets/."
    )

print("Motor importado desde Drive:")
print(f"  config.py y nucleo.py copiados de {RUTA_CODIGO_DRIVE}")
print(f"  Clases multiclase: {config.CLASES_MULTICLASE}")


In [ ]:
from pathlib import Path

# Una sola arquitectura y una sola semilla para esta primera corrida.
# Para comparar arquitecturas más adelante, alcanza con agregar más nombres
# a esta lista y volver a correr (lo ya entrenado se reutiliza).
ARQUITECTURAS = ["efficientnet_b0"]
SEMILLAS = [42]

# Rutas: datos en disco local (rápido), resultados en Drive (persisten)
config.RUTA_PARTICIONES_MULTICLASE = Path(RUTA_DATOS_LOCAL)
config.RUTA_RESULTADOS = Path('/content/drive/MyDrive/Tesis/Resultados2')
config.RUTA_MODELOS = config.RUTA_RESULTADOS / "modelos"
config.RUTA_METRICAS = config.RUTA_RESULTADOS / "metricas"
config.preparar_carpetas()

config.NUM_WORKERS = 4   # Colab tiene más CPU que tu portátil. No afecta el resultado.
config.BATCH_SIZE = 16

print(f"Arquitecturas a entrenar: {ARQUITECTURAS}")
print(f"Semillas: {SEMILLAS}")
print(f"Dispositivo: {config.DISPOSITIVO} | Batch: {config.BATCH_SIZE}")
print("\nParámetros heredados de tu config.py (idénticos a local):")
print(f"  learning_rate ....... {config.LEARNING_RATE}")
print(f"  weight_decay ........ {config.WEIGHT_DECAY}")
print(f"  epocas_max .......... {config.EPOCAS}")
print(f"  paciencia ........... {config.PACIENCIA_EARLY_STOPPING}")
print(f"  preentrenado ........ {config.PREENTRENADO}")
print(f"  amp ................. {config.USAR_AMP}")
print(f"  aumento_robustez .... {config.AUMENTO_ROBUSTEZ}")
print(f"  clases multiclase ... {config.CLASES_MULTICLASE}")

if not config.AUMENTO_ROBUSTEZ:
    print("\n  [AVISO] AUMENTO_ROBUSTEZ = False en tu config.py: entrenarías SIN")
    print("  blur ni JPEG. Si quieres el pipeline anti-atajo, ponlo en True.")


In [ ]:
import json, statistics
from datetime import datetime
from pathlib import Path

RUTA_PARCIALES = config.RUTA_METRICAS / "parciales"
RUTA_PARCIALES.mkdir(parents=True, exist_ok=True)

resumen_final = {}   # arquitectura -> (media_acc, media_f1_macro)

for arquitectura in ARQUITECTURAS:
    config.MODELO = arquitectura

    print("\n" + "=" * 64)
    print(f"EXPERIMENTO MULTICLASE — {arquitectura}")
    print("=" * 64)

    resultados = []
    for semilla in SEMILLAS:
        parcial = RUTA_PARCIALES / f"experimentoMulticlase_{arquitectura}_semilla{semilla}.json"
        if parcial.exists():
            with open(parcial, encoding="utf-8") as f:
                resultados.append(json.load(f))
            print(f"  [semilla {semilla}] ya estaba hecha — la reutilizo")
            continue

        nombre = f"multiclase_{arquitectura}_semilla{semilla}.pth"
        ruta_pth = config.RUTA_MODELOS / nombre

        # El entrenamiento real lo hace tu nucleo.py, sin modificar
        r = nucleo.entrenar_modelo_multiclase(semilla, ruta_pth)
        r["ruta_modelo"] = str(ruta_pth)
        resultados.append(r)

        with open(parcial, "w", encoding="utf-8") as f:
            json.dump(r, f, indent=2, ensure_ascii=False)
        print(f"    -> semilla {semilla} guardada en Drive")

    # --- JSON final de esta arquitectura ---
    salida = {
        "experimento": "multiclase",
        "modelo": arquitectura,
        "clases": config.CLASES_MULTICLASE,
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "config": {
            "batch_size": config.BATCH_SIZE,
            "learning_rate": config.LEARNING_RATE,
            "weight_decay": config.WEIGHT_DECAY,
            "epocas_max": config.EPOCAS,
            "paciencia": config.PACIENCIA_EARLY_STOPPING,
            "preentrenado": config.PREENTRENADO,
            "amp": config.USAR_AMP,
        },
        "resultados_por_semilla": resultados,
    }
    ruta_json = config.RUTA_METRICAS / f"experimentoMulticlase_{arquitectura}.json"
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(salida, f, indent=2, ensure_ascii=False)

    accs = [r["metricas_test"]["accuracy"] for r in resultados]
    f1s = [r["metricas_test"]["f1_macro"] for r in resultados]
    resumen_final[arquitectura] = (statistics.mean(accs), statistics.mean(f1s))

    print(f"  {'Semilla':>8} | {'Acc':>7} | {'F1 macro':>9}")
    for r in resultados:
        m = r["metricas_test"]
        print(f"  {r['semilla']:>8} | {m['accuracy']:.4f} | {m['f1_macro']:>9.4f}")
    print(f"  -> {ruta_json}")

# --- Resumen ---
print("\n" + "=" * 64)
print("TERMINADO — Experimento multiclase")
print("=" * 64)
print(f"{'Arquitectura':<18} | {'Acc media':>10} | {'F1 macro media':>15}")
print("-" * 50)
for arq, (acc, f1) in resumen_final.items():
    print(f"{arq:<18} | {acc:>10.4f} | {f1:>15.4f}")


---
## Cuando termine

En Drive tendrás `Tesis/Resultados2/`:
- `metricas/experimentoMulticlase_efficientnet_b0.json`
- `modelos/multiclase_efficientnet_b0_semilla42.pth`

**Siguiente paso:** bajá el `.pth` a tu carpeta local `Resultados/modelos/multiclase/` y la página web (`webapp/app.py`) lo va a detectar solo (`inferencia_multiclase.encontrar_modelo` toma el más reciente si no le indicás uno explícito).

**Si más adelante querés comparar arquitecturas o promediar varias semillas** (por ejemplo para el capítulo de resultados de la tesis), alcanza con ampliar `ARQUITECTURAS` y/o `SEMILLAS` en la Celda 5 y volver a correr: lo ya entrenado con esta corrida se reutiliza tal cual.
